### 1. 基础API的使用
1.1 put()/get()：写入/读取API

举例1：基于InMemoryStore

添加

In [2]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

namespace = ("users",)
user_id = "user-1"
username = "小兰"
store.put(namespace, user_id, {"name": username})

print(store.get(namespace, user_id))

Item(namespace=['users'], key='user-1', value={'name': '小兰'}, created_at='2026-10-12T15:30:13.865060+00:00', updated_at='2026-10-12T15:30:13.865060+00:00')


更新:对同一条数据进行更改后查询

In [3]:
store.put(namespace, user_id, {"name": "小红"})
print(store.get(namespace, user_id))

Item(namespace=['users'], key='user-1', value={'name': '小红'}, created_at='2026-10-12T15:40:26.744111+00:00', updated_at='2026-10-12T15:40:26.744111+00:00')


举例2：基于PostgresStore

添加

In [9]:
from langgraph.store.postgres import PostgresStore
from dotenv import load_dotenv
import os

load_dotenv(override=True)

DB_IP = os.getenv("POSTGRE_DB_IP")
DB_USER = os.getenv("POSTGRE_DB_USER")
DB_PWD = os.getenv("POSTGRE_DB_PWD")

DB_URL = f"postgresql://{DB_USER}:{DB_PWD}@{DB_IP}:5432/langchain_memory?sslmode=disable"

namespace = ("users",)
user_id = "user-1"
username = "兰陵王"

with PostgresStore.from_conn_string(DB_URL) as postgresql_store:
    postgresql_store.setup()
    postgresql_store.put(namespace, user_id, {"name": username})
    print(postgresql_store.get(namespace, user_id))


Item(namespace=['users'], key='user-1', value={'name': '兰陵王'}, created_at='2026-10-12T23:49:28.179533+08:00', updated_at='2026-10-12T23:49:45.688943+08:00')


更新

In [10]:
with PostgresStore.from_conn_string(DB_URL) as postgresql_store:
    postgresql_store.setup()
    postgresql_store.put(namespace, user_id, {"name": "典韦"})
    print(postgresql_store.get(namespace, user_id))

Item(namespace=['users'], key='user-1', value={'name': '典韦'}, created_at='2026-10-12T23:49:28.179533+08:00', updated_at='2026-10-12T23:51:43.770477+08:00')


1.2 search()：检索API

举例1：按照namespace前缀搜索

In [11]:
from langgraph.store.memory import InMemoryStore
store = InMemoryStore()
namespace1 = ("users", "Alice", "memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}

namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}

namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}
store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)

In [12]:
print('=' * 30, '-> (users, ) <-', "=" * 30)
for item in store.search(("users", )):
    print(item)

============================== -> (users, ) <- ==============================
Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-10-12T16:11:19.967556+00:00', updated_at='2026-10-12T16:11:19.967556+00:00', score=None)
Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-10-12T16:11:19.967556+00:00', updated_at='2026-10-12T16:11:19.967556+00:00', score=None)
Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-10-12T16:11:19.967556+00:00', updated_at='2026-10-12T16:11:19.967556+00:00', score=None)


In [13]:
print('=' * 30, '-> (users, Alice) <-', "=" * 30)
for item in store.search(("users", "Alice")):
    print(item)

============================== -> (users, Alice) <- ==============================
Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-10-12T16:11:19.967556+00:00', updated_at='2026-10-12T16:11:19.967556+00:00', score=None)


举例2：按照filter过滤

In [14]:
print("=" * 30, '-> (users, ), filter=food <-", "=" * 30)')
for item in store.search(("users", ), filter={"food": "紫光园奶皮子酸奶"}):
    print(item)

============================== -> (users, ), filter=food <-", "=" * 30)
Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-10-12T16:11:19.967556+00:00', updated_at='2026-10-12T16:11:19.967556+00:00', score=None)
Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-10-12T16:11:19.967556+00:00', updated_at='2026-10-12T16:11:19.967556+00:00', score=None)


In [15]:
print("=" * 30, '-> (users, ), filter=sports <-", "=" * 30)')
for item in store.search(("users", ), filter={"sports": "跑步"}):
    print(item)

============================== -> (users, ), filter=sports <-", "=" * 30)
Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-10-12T16:11:19.967556+00:00', updated_at='2026-10-12T16:11:19.967556+00:00', score=None)
Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-10-12T16:11:19.967556+00:00', updated_at='2026-10-12T16:11:19.967556+00:00', score=None)


In [16]:
print("=" * 30, '-> (users, ), filter=course <-", "=" * 30)')
for item in store.search(("users", ), filter={"course": "数字电路与模拟电路"}):
    print(item)

============================== -> (users, ), filter=course <-", "=" * 30)
Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-10-12T16:11:19.967556+00:00', updated_at='2026-10-12T16:11:19.967556+00:00', score=None)
Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-10-12T16:11:19.967556+00:00', updated_at='2026-10-12T16:11:19.967556+00:00', score=None)


举例3：按照语义搜索

- embeds ：将输入文本转换为向量的嵌入函数，可以是 自定义函数 ，也可以是 嵌入模型对象 ，本例传递的是自定义嵌入函数。
- dims ：输出向量维度
- fields ：用于计算向量的属性列表，这里的属性都是指value中的key，value是一个JSON-like字典，可取值如下
  - ["$"] ：将value作为整体嵌入
  - ["fileds1", "fields2"] ：单独指定某个一级字段
  - ["parent.child"] ：从内部的嵌套JSON对象中获取子字段的值
  - ["array[*].field"] ：从JSON数组的每个JSON对象中获取子字段的值

   - 注意：上述四种形式可以同时出现，fields列表的每个元素都会生成一个嵌入向量。

举例1：自定义嵌入模型

In [17]:
from langgraph.store.memory import InMemoryStore

def embed(text: list[str]) -> list[list[float]]:
    return [[1.0] * 6 for _ in range(len(text))]

index_config = {
    "embed": embed,
    "dim": 6,
    "fields": ["$", "course"]
}

store = InMemoryStore(
    index=index_config,
)

namespace1 = ("users", "Alice", "memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}

namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}

namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}

store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)

In [18]:
from pprint import pprint
pprint(store._vectors)

defaultdict(<function InMemoryStore.__init__.<locals>.<lambda> at 0x0000021685C86B60>,
            {('users', 'Alice', 'memories'): defaultdict(<class 'dict'>,
                                                         {'preferences': {'$': [1.0,
                                                                                1.0,
                                                                                1.0,
                                                                                1.0,
                                                                                1.0,
                                                                                1.0],
                                                                          'course': [1.0,
                                                                                     1.0,
                                                                                     1.0,
                                                           

In [19]:
from pprint import pprint
pprint(store._vectors[('users', 'Alice', 'memories')]['preferences']['$'])

[1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


举例2：使用嵌入模型

In [24]:
from langgraph.store.memory import InMemoryStore
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
import os

load_dotenv(override=True)
embedding_model = OpenAIEmbeddings(
    model="qwen3.7-text-embedding",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    check_embedding_ctx_length=False,  # 建议加上此参数以避免 token 化问题
)

index_config = {
    "embed": embedding_model,
    "dim": 3072,
    "fields": ["$"]
}
store = InMemoryStore(
    index=index_config,
)

namespace1 = ("users", "Alice", "memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}

namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}

namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}

store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)

In [25]:
for item in store.search(("users", ), query="数电模电"):
    print(item)

Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-10-12T16:59:53.104559+00:00', updated_at='2026-10-12T16:59:53.104559+00:00', score=0.6370821358509399)
Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-10-12T16:59:53.221098+00:00', updated_at='2026-10-12T16:59:53.221098+00:00', score=0.5925621094150559)
Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-10-12T16:59:52.986996+00:00', updated_at='2026-10-12T16:59:52.986996+00:00', score=0.42467846604956744)
